# MPMAvatar — SDS Physics Hypothesis Test
**Lightning AI Studio Notebook**

Pipeline:
1. Clone repo from GitHub
2. Install dependencies + build CUDA extensions
3. Download Wan 2.2 I2V checkpoint
4. Upload / mount your trained Gaussians
5. Run `train_sds_physics.py`

---
**GPU required:** A100 / A10G (40GB+) recommended for Wan 5B

## Cell 0 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set CUDA arch for custom extension builds
import os
cc = torch.cuda.get_device_capability()
arch = f"{cc[0]}{cc[1]}"
os.environ['TORCH_CUDA_ARCH_LIST'] = f"{cc[0]}.{cc[1]}"
os.environ['FORCE_CUDA'] = '1'
print(f"CUDA arch: sm_{arch}  →  TORCH_CUDA_ARCH_LIST={os.environ['TORCH_CUDA_ARCH_LIST']}")

## Cell 1 — Clone Repo

In [ ]:
import os

# ── CHANGE THIS to your GitHub repo URL ──────────────────────────────────────
GITHUB_REPO = "https://github.com/YOUR_USERNAME/MPMAvatar.git"
REPO_DIR    = "/teamspace/studios/this_studio/MPMAvatar"
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}, pulling latest...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

## Cell 2 — Install Python Dependencies

In [ ]:
# Core dependencies
!pip install -q \
    accelerate==0.25.0 \
    wandb \
    roma==1.5.2.1 \
    jaxtyping==0.2.28 \
    einops==0.7.0 \
    plyfile==1.0.3 \
    trimesh==4.0.8 \
    smplx==0.1.28 \
    tqdm \
    pyyaml \
    numpy==1.25.0 \
    scipy \
    Pillow \
    mediapy \
    imageio==2.34.0 \
    safetensors==0.3.3

# Warp (MPM solver)
!pip install -q warp-lang==0.10.1

# Transformers (for T5 tokenizer)
!pip install -q transformers sentencepiece

# HuggingFace Hub (for downloading Wan)
!pip install -q huggingface_hub

print("Done.")

## Cell 3 — Build CUDA Extensions

In [ ]:
import os
os.chdir(REPO_DIR)

# ── diff-gaussian-rasterization ───────────────────────────────────────────────
EXT_DIR = "/teamspace/studios/this_studio/ext"
os.makedirs(EXT_DIR, exist_ok=True)

diff_gauss_dir = f"{EXT_DIR}/diff-gaussian-rasterization"
if not os.path.exists(diff_gauss_dir):
    !git clone --recursive \
        https://github.com/slothfulxtx/diff-gaussian-rasterization.git \
        {diff_gauss_dir}

!cd {diff_gauss_dir} && pip install -e . --quiet
print("diff-gaussian-rasterization installed")

# ── simple-knn ────────────────────────────────────────────────────────────────
simple_knn_dir = f"{EXT_DIR}/simple-knn"
if not os.path.exists(simple_knn_dir):
    !git clone https://gitlab.inria.fr/bkerbl/simple-knn.git {simple_knn_dir}

!cd {simple_knn_dir} && pip install -e . --quiet
print("simple-knn installed")

# ── Verify ────────────────────────────────────────────────────────────────────
try:
    import diff_gauss
    print("✓ diff_gauss OK")
except ImportError as e:
    print(f"✗ diff_gauss FAILED: {e}")

try:
    import simple_knn
    print("✓ simple_knn OK")
except ImportError as e:
    print(f"✗ simple_knn FAILED: {e}")

## Cell 4 — Install Wan 2.2 Repo

In [ ]:
WAN_REPO_DIR = "/teamspace/studios/this_studio/Wan2.2"

if not os.path.exists(WAN_REPO_DIR):
    !git clone https://github.com/Wan-Video/Wan2.2.git {WAN_REPO_DIR}

!cd {WAN_REPO_DIR} && pip install -e . --quiet

# Verify wan imports
import sys
sys.path.insert(0, WAN_REPO_DIR)
try:
    from wan.modules.model import WanModel
    from wan.modules.vae2_1 import Wan2_1_VAE
    print("✓ Wan modules importable")
except ImportError as e:
    print(f"✗ Wan import failed: {e}")

## Cell 5 — Download Wan 2.2 I2V Checkpoint

> ⚠️ This is **~40 GB**. Run once, stored persistently in Lightning teamspace.
> Skip if already downloaded.

In [ ]:
WAN_CKPT_DIR = "/teamspace/studios/this_studio/wan_checkpoints"

if not os.path.exists(WAN_CKPT_DIR):
    os.makedirs(WAN_CKPT_DIR, exist_ok=True)

    # ── Option A: HuggingFace Hub snapshot (recommended) ─────────────────────
    from huggingface_hub import snapshot_download

    print("Downloading Wan 2.2 I2V 14B (~40GB). This will take a while...")
    snapshot_download(
        repo_id="Wan-AI/Wan2.2-I2V-14B-480P",
        local_dir=WAN_CKPT_DIR,
        ignore_patterns=["*.md", "*.txt"],
    )
    print(f"Downloaded to {WAN_CKPT_DIR}")
else:
    print(f"Checkpoint already exists at {WAN_CKPT_DIR}")

# Verify structure
!ls -lh {WAN_CKPT_DIR}

## Cell 6 — Pre-download T5 Tokenizer

In [ ]:
T5_CACHE = "/teamspace/studios/this_studio/t5_cache"
os.makedirs(T5_CACHE, exist_ok=True)
os.environ['TRANSFORMERS_CACHE'] = T5_CACHE
os.environ['HF_HOME']            = T5_CACHE

from transformers import AutoTokenizer
print("Downloading umt5-xxl tokenizer (~2GB)...")
AutoTokenizer.from_pretrained('google/umt5-xxl', cache_dir=T5_CACHE)
print("Tokenizer ready.")

## Cell 7 — Upload Trained Gaussians

Upload your trained Gaussian model from Stage 1 (appearance training).

**Option A:** Upload via Lightning Studio file browser (drag and drop)

**Option B:** Upload from local machine using the cell below

Expected structure:
```
/teamspace/studios/this_studio/gaussians/
├── point_cloud/
│   └── iteration_XXXXX/
│       └── point_cloud.ply
├── verts_offset.npy
├── cams.npz
└── shadow_net.pt
```

In [ ]:
GAUSSIAN_DIR = "/teamspace/studios/this_studio/gaussians"

# Check what's there
!ls -lh {GAUSSIAN_DIR}/point_cloud/ 2>/dev/null || echo "No gaussians found — upload them first"

# Verify key files exist
import os
checks = [
    f"{GAUSSIAN_DIR}/verts_offset.npy",
    f"{GAUSSIAN_DIR}/cams.npz",
    f"{GAUSSIAN_DIR}/shadow_net.pt",
]
for f in checks:
    status = "✓" if os.path.exists(f) else "✗ MISSING"
    print(f"{status}  {f}")

## Cell 8 — Upload Dataset Files

Upload your `split_idx.npz` and SMPLX fitted params.

Expected structure:
```
/teamspace/studios/this_studio/data/
├── split_idx.npz
├── a1_s1/
│   └── smplx_fitted/
│       ├── 000460.pth
│       ├── 000461.pth
│       └── ...
└── body_models/
    └── smplx/
```

In [ ]:
DATA_DIR = "/teamspace/studios/this_studio/data"

checks = [
    f"{DATA_DIR}/split_idx.npz",
]
for f in checks:
    status = "✓" if os.path.exists(f) else "✗ MISSING"
    print(f"{status}  {f}")

## Cell 9 — Config

Set all paths here. Everything else uses these variables.

In [ ]:
import os
os.chdir(REPO_DIR)

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAINED_MODEL_PATH = "/teamspace/studios/this_studio/gaussians"    # Stage 1 output
MODEL_PATH         = "/teamspace/studios/this_studio/gaussians"    # same as above usually
DATASET_DIR        = "/teamspace/studios/this_studio/data"
SPLIT_IDX_PATH     = "/teamspace/studios/this_studio/data/split_idx.npz"
OUTPUT_DIR         = "/teamspace/studios/this_studio/output"
WAN_CKPT_DIR       = "/teamspace/studios/this_studio/wan_checkpoints"
WAN_REPO_ROOT      = "/teamspace/studios/this_studio/Wan2.2"
SDS_CFG            = f"{REPO_DIR}/bridge_sds/configs/sds_test.yaml"

# ── Dataset config ─────────────────────────────────────────────────────────────
ACTOR              = 1        # ActorsHQ actor number
SEQUENCE           = 1        # ActorsHQ sequence number
TRAIN_FRAME_START  = 460      # first frame
TRAIN_FRAME_NUM    = 10       # number of frames to use
VERTS_START_IDX    = 460      # same as TRAIN_FRAME_START usually

# ── Training ───────────────────────────────────────────────────────────────────
ITERATIONS         = 100
WANDB_PROJECT      = "MPMAvatar_SDS"

# ── WandB (optional) ──────────────────────────────────────────────────────────
USE_WANDB          = True     # set False to disable
WANDB_API_KEY      = ""       # paste your key or leave empty (will prompt)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config set:")
print(f"  Repo          : {REPO_DIR}")
print(f"  Gaussians     : {TRAINED_MODEL_PATH}")
print(f"  Dataset       : {DATASET_DIR}")
print(f"  Wan ckpt      : {WAN_CKPT_DIR}")
print(f"  Wan repo      : {WAN_REPO_ROOT}")
print(f"  SDS config    : {SDS_CFG}")
print(f"  Iterations    : {ITERATIONS}")

## Cell 10 — WandB Login (optional)

In [ ]:
if USE_WANDB:
    import wandb
    if WANDB_API_KEY:
        wandb.login(key=WANDB_API_KEY)
    else:
        wandb.login()  # interactive prompt
    print("WandB logged in")
else:
    import os
    os.environ['WANDB_MODE'] = 'disabled'
    print("WandB disabled")

## Cell 11 — Verify Everything Before Running

In [ ]:
import os

checks = {
    "Repo":           REPO_DIR,
    "train_sds_physics.py": f"{REPO_DIR}/train_sds_physics.py",
    "bridge_sds/wan22": f"{REPO_DIR}/bridge_sds/wan22_i2v_guidance.py",
    "SDS config yaml": SDS_CFG,
    "Gaussians dir":  TRAINED_MODEL_PATH,
    "split_idx.npz":  SPLIT_IDX_PATH,
    "Wan ckpt dir":   WAN_CKPT_DIR,
    "Wan repo":       WAN_REPO_ROOT,
    "Dataset dir":    DATASET_DIR,
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ MISSING"
    if not exists:
        all_ok = False
    print(f"{status}  {name}: {path}")

print()
if all_ok:
    print("✓ All checks passed — ready to run")
else:
    print("✗ Fix missing paths above before running")

## Cell 12 — Run SDS Physics Training

In [ ]:
import os
import sys

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, WAN_REPO_ROOT)

# Set tokenizer cache so it finds the pre-downloaded T5
os.environ['TRANSFORMERS_CACHE'] = T5_CACHE
os.environ['HF_HOME']            = T5_CACHE

wandb_flag = "--use_wandb" if USE_WANDB else ""

cmd = f"""
python train_sds_physics.py \\
    --trained_model_path {TRAINED_MODEL_PATH} \\
    --model_path         {MODEL_PATH} \\
    --dataset_dir        {DATASET_DIR} \\
    --split_idx_path     {SPLIT_IDX_PATH} \\
    --output_dir         {OUTPUT_DIR} \\
    --actor              {ACTOR} \\
    --sequence           {SEQUENCE} \\
    --train_frame_start_num {TRAIN_FRAME_START} {TRAIN_FRAME_NUM} \\
    --verts_start_idx    {VERTS_START_IDX} \\
    --wan_ckpt_dir       {WAN_CKPT_DIR} \\
    --wan_repo_root      {WAN_REPO_ROOT} \\
    --sds_cfg            {SDS_CFG} \\
    --iterations         {ITERATIONS} \\
    --save_name          sds_test \\
    {wandb_flag} \\
    --wandb_project      {WANDB_PROJECT}
""".strip()

print("Running command:")
print(cmd)
print("\n" + "="*60 + "\n")

os.system(cmd)

## Cell 13 — Monitor Output Files

In [ ]:
import os
import pandas as pd

# Find param trajectory CSV
csv_candidates = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        if f == "param_trajectory.csv":
            csv_candidates.append(os.path.join(root, f))

if csv_candidates:
    csv_path = csv_candidates[-1]
    print(f"Found: {csv_path}")
    df = pd.read_csv(csv_path)
    print(df.tail(20).to_string())
else:
    print("No param_trajectory.csv found yet")

## Cell 14 — Plot Parameter Trajectories

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if csv_candidates:
    df = pd.read_csv(csv_candidates[-1])

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("SDS Physics Parameter Optimization", fontsize=14)

    # Parameters
    axes[0,0].plot(df['step'], df['D'],        'b-o', ms=3); axes[0,0].set_title('Density D')
    axes[0,1].plot(df['step'], df['E_Pa'],     'r-o', ms=3); axes[0,1].set_title('Young\'s E (Pa)')
    axes[0,2].plot(df['step'], df['H'],        'g-o', ms=3); axes[0,2].set_title('Height Scale H')
    axes[1,0].plot(df['step'], df['friction'], 'm-o', ms=3); axes[1,0].set_title('Friction')

    # Loss
    axes[1,1].plot(df['step'], df['sds_loss_base'], 'k-o', ms=3); axes[1,1].set_title('SDS Loss (base)')

    # Gradients
    axes[1,2].plot(df['step'], df['grad_D'], label='D')
    axes[1,2].plot(df['step'], df['grad_E'], label='E')
    axes[1,2].plot(df['step'], df['grad_H'], label='H')
    axes[1,2].set_title('SPSA Gradients')
    axes[1,2].legend()

    for ax in axes.flat:
        ax.set_xlabel('step')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/param_trajectory.png", dpi=150)
    plt.show()
    print(f"Saved to {OUTPUT_DIR}/param_trajectory.png")
else:
    print("No CSV data yet")

## Cell 15 — Show Best Params Found

In [ ]:
import numpy as np
import glob

npz_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/sds_best_param_*.npz", recursive=True))

if npz_files:
    latest = npz_files[-1]
    print(f"Latest checkpoint: {latest}")
    data = np.load(latest)
    print("\nBest parameters found:")
    for k, v in data.items():
        print(f"  {k:12s} = {float(v):.6f}")
else:
    print("No checkpoint files found yet")